# Energies for [Co(H2O)5CO2]^2+ (doublet / quartet)

Reproduces paper Figures 9, 10 (gas) and 11, 12 (solvent):
potential energy curves and doublet-quartet gaps for ROHF, HCI, UCCSD(T) and SQD at 34 qubits.

Input: `./data/energies_charge2.csv`. Output: `./figures/`.

### Imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick


import numpy as np
import pandas as pd
import scienceplots

### Setup:

In [ ]:
basis_set = 'def2-TZVP'
distance = [1.44, 1.97, 2.51, 3.04, 3.57, 4.10, 4.63, 5.16, 5.69, 6.23]   # Co-O(CO2) distance [Angstrom] for shifts -0.15 ... 1.2
# ylim = (-1948.7, -1948.14)

n_qubits_list = [34]
n_elec = { # first medium, then n_qubits, then spin value
    "solvent":{
        34: {1: 23, 3: 23}
    },
    "gas":{
        34: {1: 23, 3: 23}
    }
}

# Energies (Hartree) of all methods, read from ./data/energies_charge2.csv
energies = pd.read_csv('./data/energies_charge2.csv')

def results_dict(medium):
    """Rebuild {'<METHOD>_<State>[_<N>so]': [E(shift) ...]} for one medium."""
    out = {}
    sub = energies[energies['medium'] == medium]
    for (method, state, nq), grp in sub.groupby(['method', 'state', 'n_qubits'], dropna=False, sort=False):
        key = f'{method}_{state.capitalize()}' + ('' if pd.isna(nq) else f'_{int(nq)}so')
        out[key] = grp.sort_values('distance_angstrom')['energy_hartree'].tolist()
    return out

gas_results = results_dict('gas')
df_gas = pd.DataFrame(gas_results, index=distance)

solvent_results = results_dict('solvent')
df_solvent = pd.DataFrame(solvent_results, index=distance)

### Helpers

In [ ]:
def plot_energy_differences(df, distance, n_qubits_list, n_elec, medium, if_HCI=True):
    df_diff = pd.DataFrame(index=distance, columns=None)

    fig,axs = plt.subplots(1, 1, figsize=(6.5, 5))#, dpi=150)
    l0, = plt.plot(distance, np.abs(df["ROHF_Quartet"] - df["ROHF_Doublet"]), '-o', markersize=4.5, label='ROHF')
    
    df_diff['ROHF'] = np.abs(df["ROHF_Quartet"] - df["ROHF_Doublet"])

    colormap = plt.get_cmap('tab10')
    n_colors = len(n_qubits_list)
    colors = colormap(np.linspace(0.2, 1, n_colors))
    
    l1 = None
    if if_HCI:
        l1, = plt.plot([None], [None], ':o', color='black', label=f'HCI')
    l2,  = plt.plot([None], [None], '--', color='black', label=f'(U)CCSD(T)')
    l3,  = plt.plot([None], [None], '-v', color='black', label=f'SQD')
    
    # add the legend immediately
    legend1 = axs.legend(frameon=True, 
                        handles=[l0, l1, l2, l3] if if_HCI else [l0, l2, l3],
                        loc='lower right' if medium=='solvent' else 'lower right', fontsize=10)
    as_handles = []
    for n_so in n_qubits_list:
        l, = plt.plot([None], 
                    [None], 
                    "-", 
                    color=colors[n_qubits_list.index(n_so)],
                    label=f'{n_so}-qubits ({n_so//2}o, {n_elec[medium][n_so][1]}e)')

        as_handles.append(l)
        if if_HCI:
            plt.plot(distance, 
                        np.abs(df[f'HCI_Quartet_{n_so}so'] - df[f'HCI_Doublet_{n_so}so']), 
                        ':o', 
                        color=colors[n_qubits_list.index(n_so)],
                        markersize=4.5, label=f'HCI {n_so}-nso')
            df_diff[f'HCI_{n_so}so'] = np.abs(df[f'HCI_Quartet_{n_so}so'] - df[f'HCI_Doublet_{n_so}so'])
        
        for method in ['CCSD(T)']:
            plt.plot(distance, 
                    np.abs(df[f'{method}_Quartet_{n_so}so'] - df[f'{method}_Doublet_{n_so}so']),
                    '--',
                    color=colors[n_qubits_list.index(n_so)],
                    label=None)
            df_diff[f'{method}_{n_so}so'] = np.abs(df[f'{method}_Quartet_{n_so}so'] - df[f'{method}_Doublet_{n_so}so'])
    
        plt.plot(distance, 
                np.abs(df[f'SQD_Quartet_{n_so}so'] - df[f'SQD_Doublet_{n_so}so']), 
                '-v', 
                color=colors[n_qubits_list.index(n_so)],
                markersize=4.5, label=None)
        df_diff[f'SQD_{n_so}so'] = np.abs(df[f'SQD_Quartet_{n_so}so'] - df[f'SQD_Doublet_{n_so}so'])
    
    axs.add_artist(legend1)
    # legend2 = axs.legend(frameon=True, 
    #                     loc='upper right',
    #                     handles=as_handles, fontsize=10
    #                     #    bbox_to_anchor=(1, -0.1)
    #                     )

    # axs.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f'))
    plt.grid()
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.ylabel('Energy [Ha]',fontsize=14)
    plt.xlabel('Distance [$\mathrm{\AA}$]',fontsize=14)
    plt.title(f"$[Co(H_2O)_5CO_2]^{{2+}}$ ({medium})",fontsize=14)
    plt.savefig(f"./figures/{medium}_charge_2_diff_results.pdf")#, dpi=250)
    plt.show()
    
    return df_diff


def plot_equilibrium_point(df, n_qubits_list, medium, if_hci=True):
    alpha = 0.7
    colors_4 = ['C0','C1','C2','C3']#plt.get_cmap('tab10')(np.linspace(0.2, 1, 4))
    #with plt.style.context('science'):
    #fig, axs = plt.subplots(1, 1, figsize=(5, 5), dpi=150)
    plt.figure(figsize=(5, 5))
    
    plt.plot([None], [None], ':v', color='gray', label='Doublet')
    plt.plot([None], [None], '--o', color='gray', label='Quartet')
    
    plt.plot([None], [None], '-', color=colors_4[2], label='ROHF')
    plt.plot([None], [None], '-', color=colors_4[0], label='SQD')
    if if_hci:
        plt.plot([None], [None], '-', color=colors_4[1], label='HCI')
    plt.plot([None], [None], '-', color=colors_4[3], label='(U)CCSD(T)')
    
    for spin_state in ['Doublet', 'Quartet']:
        
        plt.axhline(df.iloc[1][f'ROHF_{spin_state}'],
                    linestyle='--' if spin_state=='Quartet' else ':',
                    color=colors_4[2],
                    label=None)
    
        if if_hci:
            plt.plot(n_qubits_list, 
                    df.iloc[1][[f'HCI_{spin_state}_{n_so}so' for n_so in n_qubits_list]],
                    '--o' if spin_state=='Quartet' else ':v',
                    color=colors_4[1],
                    alpha=alpha,
                    label=None)
        plt.plot(n_qubits_list, 
                df.iloc[1][[f'CCSD(T)_{spin_state}_{n_so}so' for n_so in n_qubits_list]],
                '--X' if spin_state=='Quartet' else ':v',
                color=colors_4[3],
                alpha=alpha,
                label=None)
        plt.plot(n_qubits_list, 
                df.iloc[1][[f'SQD_{spin_state}_{n_so}so' for n_so in n_qubits_list]],
                '--^' if spin_state=='Quartet' else ':v',
                color=colors_4[0],
                alpha=alpha,
                label=None)
    
    plt.legend(frameon=True, loc='upper right')
    plt.grid()
    # if medium=='gas':
    #     plt.ylim(-1948.6, -1948.28)
    # else:
    #     pass
    #     # plt.ylim(-1948.7, -1948.14)
    #axs.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.xlabel('Number of Qubits',fontsize=14)
    plt.ylabel('Energy [Ha]',fontsize=14)
    plt.title(f"Quartet and Doublet Energies for $[Co(H_2O)_5CO_2]^{{2+}}$\n({medium})", fontsize=14)
    plt.tight_layout()
    #plt.savefig(f"./figures/{medium}_charge_2_eq_point_results.pdf")#, dpi=250)
    plt.show()
        
        
def plot_PES(df, n_qubits_list, n_elec, medium, if_HCI=True):
    #with plt.style.context('science'):
    fig,axs = plt.subplots(2, 1, figsize=(5, 7.5))#, dpi=150) # 250
    for i, n_so in enumerate(n_qubits_list):
        for j, (spin_state, spin_number) in enumerate([('Quartet', 3), ('Doublet', 1)]):
            n_e = (n_elec[medium][n_so][spin_number]//2 + spin_number//2,
                n_elec[medium][n_so][spin_number]//2 - spin_number//2)

            axs[j].plot(df[f"ROHF_{spin_state}"], '-o', label='ROHF')
            if if_HCI:
                axs[j].plot(df[f"HCI_{spin_state}_{n_so}so"], '-o', label=f'HCI'# ($\epsilon_1$={e1_det:.0e}, $\epsilon_2$={e2_det})'
                            )            
            axs[j].plot(df[f"SQD_{spin_state}_{n_so}so"], '-v', label=f'SQD')
            axs[j].plot(df[f"CCSD(T)_{spin_state}_{n_so}so"], label='(U)CCSD(T)')    
            
            axs[1].legend(frameon=True, 
                    loc='upper right',
                    # bbox_to_anchor=(1, -0.15)
                    )
            # if medium=='gas':
            #     axs[j].set_ylim(*ylim)
            axs[j].yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))

            axs[j].grid()
            
            axs[j].set_ylabel(f'E ({spin_state}) [Ha]')
            axs[1].set_xlabel('Distance [$\mathrm{\AA}$]')
            #axs[0].set_title(f"({n_so//2}o, {n_elec[medium][n_so][spin_number]}e) - {n_so}-qubits")
            axs[0].set_title(f"$[Co(H_2O)_5CO_2]^{{2+}}$ ({medium})", fontsize=14)
        
    #plt.suptitle(f"$[Co(H_2O)_5CO_2]^{{2+}}$ ({medium} phase)", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"./figures/{medium}_charge_2_results.pdf")#, dpi=250)
    plt.show()

In [ ]:
medium = 'gas'

plot_PES(df_gas, n_qubits_list, n_elec, medium)
df_diff = plot_energy_differences(df_gas, distance, n_qubits_list, n_elec, medium)
plot_equilibrium_point(df_gas, n_qubits_list, medium)

In [ ]:
medium = 'solvent'
has_hci = 'HCI_Doublet_34so' in df_solvent.columns   # HCI curves are drawn when HCI energies are present in the CSV

plot_PES(df_solvent, n_qubits_list, n_elec, medium, if_HCI=has_hci)
df_diff = plot_energy_differences(df_solvent, distance, n_qubits_list, n_elec, medium, if_HCI=has_hci)
plot_equilibrium_point(df_solvent, n_qubits_list, medium, if_hci=has_hci)